In [1]:
import os
import gc
import psutil
import pyarrow as pa
import pyarrow.parquet as pq

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

processo = psutil.Process(os.getpid())
pool = pa.default_memory_pool()

def memoria(etapa):
    m = processo.memory_full_info()

    print(f"\n{etapa}")
    print(f"RSS: {m.rss / 1024**2:.2f} MB")
    print(f"USS: {m.uss / 1024**2:.2f} MB")
    print(f"Arrow: {pool.bytes_allocated() / 1024**2:.2f} MB")


memoria("INICIAL")

# 1. Abre o arquivo e lê somente os metadados
parquet = pq.ParquetFile(arquivo)

memoria("APÓS ABRIR PARQUET")

# 2. Lê o único row group
tabela = parquet.read_row_group(0)

memoria("APÓS LER ROW GROUP")

print("\nTamanho dos buffers:", tabela.nbytes / 1024**2, "MB")


INICIAL
RSS: 96.68 MB
USS: 48.58 MB
Arrow: 0.00 MB

APÓS ABRIR PARQUET
RSS: 103.48 MB
USS: 52.62 MB
Arrow: 0.00 MB

APÓS LER ROW GROUP
RSS: 263.99 MB
USS: 209.47 MB
Arrow: 64.07 MB

Tamanho dos buffers: 55.50620365142822 MB


In [1]:
import psutil
import pyarrow as pa
import pyarrow.parquet as pq

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

processo = psutil.Process()
pool = pa.default_memory_pool()

def memoria(etapa, tabela=None):
    m = processo.memory_full_info()

    print(f"\n{etapa}")
    print(f"RSS: {m.rss / 1024**2:.2f} MB")
    print(f"USS: {m.uss / 1024**2:.2f} MB")
    print(f"Arrow: {pool.bytes_allocated() / 1024**2:.2f} MB")

    if tabela is not None:
        print(f"Buffers: {tabela.nbytes / 1024**2:.2f} MB")


parquet = pq.ParquetFile(arquivo)

memoria("ANTES DA LEITURA")

colunas = [
    "ano",
    "CO_MUNICIPIO_TERRITORIO",
    "TIPO_REFERENCIA_TERRITORIAL",
    "tipo_grupo",
    "grupo",
    "disciplina",
    "media",
    "n_validos",
    "mediana",
    "std",
    "q1",
    "q3",
]

tabela = parquet.read_row_group(0, columns=colunas)

memoria("APÓS LEITURA DAS 12 COLUNAS", tabela)

print(f"\nTamanho dos buffers: {tabela.nbytes / 1024**2:.2f} MB")


ANTES DA LEITURA
RSS: 103.40 MB
USS: 52.54 MB
Arrow: 0.00 MB

APÓS LEITURA DAS 12 COLUNAS
RSS: 233.88 MB
USS: 179.37 MB
Arrow: 46.73 MB
Buffers: 40.13 MB

Tamanho dos buffers: 40.13 MB


In [1]:
import psutil
import pyarrow as pa
import pyarrow.parquet as pq

arquivo = "data/consumo/uf=PA/desempenho_grupos_municipal.parquet"

processo = psutil.Process()
pool = pa.default_memory_pool()


def memoria(etapa, tabela=None):
    m = processo.memory_full_info()

    print(f"\n{etapa}")
    print(f"RSS: {m.rss / 1024**2:.2f} MB")
    print(f"USS: {m.uss / 1024**2:.2f} MB")
    print(f"Arrow: {pool.bytes_allocated() / 1024**2:.2f} MB")
    print(f"Pico Arrow: {pool.max_memory() / 1024**2:.2f} MB")

    if tabela is not None:
        print(f"Buffers: {tabela.nbytes / 1024**2:.2f} MB")


# Abre o arquivo
parquet = pq.ParquetFile(arquivo)

memoria("ANTES DA LEITURA")

# Seleção de sete colunas
colunas = [
    "ano",
    "media",
    "n_validos",
    "mediana",
    "std",
    "q1",
    "q3",
]

# Leitura do Row Group
tabela = parquet.read_row_group(
    0,
    columns=colunas
)

memoria("APÓS LEITURA", tabela)

print(f"\nTamanho dos buffers: {tabela.nbytes / 1024**2:.2f} MB")
print(f"Linhas: {tabela.num_rows:,}")
print(f"Colunas: {tabela.num_columns}")


ANTES DA LEITURA
RSS: 103.46 MB
USS: 52.57 MB
Arrow: 0.00 MB
Pico Arrow: 0.06 MB

APÓS LEITURA
RSS: 164.80 MB
USS: 111.26 MB
Arrow: 24.52 MB
Pico Arrow: 37.71 MB
Buffers: 18.29 MB

Tamanho dos buffers: 18.29 MB
Linhas: 337,214
Colunas: 7
